# XGBoost Classifier: Predicting Intraday Return Direction from One-Year Momentum

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve

## Load Data

In [ ]:
with open("sp500_data.pkl", "rb") as f:
    data = pickle.load(f)

adj_close        = data["adj_close"]          # DataFrame: dates x tickers
intraday_returns = data["intraday_returns"]    # DataFrame: dates x tickers

print("adj_close shape:", adj_close.shape)
print("intraday_returns shape:", intraday_returns.shape)
print("Date range:", adj_close.index[0].date(), "→", adj_close.index[-1].date())

## Feature Engineering: One-Year Momentum

In [ ]:
# One-year momentum: trailing 252-day return, skipping the most recent 21 days
# (standard Jegadeesh-Titman definition to avoid short-term reversal contamination)
one_year_momentum = (adj_close.shift(21) / adj_close.shift(252)) - 1

# Intraday momentum: previous day's intraday return (1-day lag to avoid lookahead bias)
intraday_momentum = intraday_returns.shift(1)

# Binary target: 1 if today's intraday return > 0, else 0
target = (intraday_returns > 0).astype(int)

print("One-year momentum NaN fraction:", one_year_momentum.isna().mean().mean().round(4))
print("Intraday momentum NaN fraction:", intraday_momentum.isna().mean().mean().round(4))
print("Target class balance (mean):", target.mean().mean().round(4))

## Build Panel Dataset

In [ ]:
# Stack wide DataFrames into (date, ticker) panel
X_oym   = one_year_momentum.stack(future_stack=True).rename("one_year_momentum")
X_idm   = intraday_momentum.stack(future_stack=True).rename("intraday_momentum")
y_panel = target.stack(future_stack=True).rename("intraday_up")

panel = pd.concat([X_oym, X_idm, y_panel], axis=1).dropna()
panel.index.names = ["date", "ticker"]

print("Panel shape:", panel.shape)
print(panel.head())

## Train / Test Split (Time-Series)

In [ ]:
# Define time-series cutoff: 80% train / 20% test by unique trading dates
panel_sorted = panel.sort_index(level="date")
dates = panel_sorted.index.get_level_values("date").unique().sort_values()

cutoff = dates[int(len(dates) * 0.80)]
print(f"Cutoff date : {cutoff.date()}")
print(f"Train dates : {dates[0].date()} → {cutoff.date()}")
print(f"Test dates  : {dates[dates > cutoff][0].date()} → {dates[-1].date()}")

## Train XGBoost Classifier

In [ ]:
FEATURES = ["intraday_momentum", "one_year_momentum"]

ticker_models  = {}   # ticker -> fitted clf
ticker_results = {}   # ticker -> dict with metrics + per-date predictions

tickers = panel.index.get_level_values("ticker").unique()
print(f"Training per-ticker XGBoost for {len(tickers)} tickers ...\n")

for i, ticker in enumerate(tickers):
    df = panel.xs(ticker, level="ticker").sort_index()

    train = df[df.index <= cutoff]
    test  = df[df.index >  cutoff]

    if len(train) < 100 or len(test) < 20:
        continue

    X_tr, y_tr = train[FEATURES], train["intraday_up"]
    X_te, y_te = test[FEATURES],  test["intraday_up"]

    scale_pw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1.0,
        scale_pos_weight=scale_pw,
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        verbosity=0,
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    y_pred      = clf.predict(X_te)
    y_pred_prob = clf.predict_proba(X_te)[:, 1]

    try:
        auc = roc_auc_score(y_te, y_pred_prob)
    except Exception:
        auc = np.nan

    ticker_models[ticker] = clf
    ticker_results[ticker] = {
        "accuracy":    accuracy_score(y_te, y_pred),
        "roc_auc":     auc,
        "predictions": pd.Series(y_pred,     index=test.index, name=ticker),
        "probas":      pd.Series(y_pred_prob, index=test.index, name=ticker),
        "y_true":      y_te,
    }

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(tickers)} tickers done")

print(f"\nFinished. Models trained for {len(ticker_models)} tickers.")

## Evaluation

In [ ]:
results_df = pd.DataFrame(
    {t: {"accuracy": v["accuracy"], "roc_auc": v["roc_auc"]}
     for t, v in ticker_results.items()}
).T

print("=== Per-Ticker Metrics (aggregated across all tickers) ===\n")
print(results_df.describe().round(4))
print(f"\nMedian Accuracy : {results_df['accuracy'].median():.4f}")
print(f"Median ROC-AUC  : {results_df['roc_auc'].median():.4f}")
print(f"Tickers with AUC > 0.52 : {(results_df['roc_auc'] > 0.52).sum()} / {len(results_df)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Accuracy distribution across tickers ---
axes[0].hist(results_df["accuracy"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(0.5, color="red", linestyle="--", label="Random (0.50)")
axes[0].axvline(results_df["accuracy"].median(), color="orange", linestyle="-",
                label=f"Median = {results_df['accuracy'].median():.3f}")
axes[0].set_xlabel("Accuracy")
axes[0].set_ylabel("# Tickers")
axes[0].set_title("Per-Ticker Accuracy Distribution")
axes[0].legend()

# --- ROC-AUC distribution across tickers ---
axes[1].hist(results_df["roc_auc"].dropna(), bins=30, color="steelblue", edgecolor="white")
axes[1].axvline(0.5, color="red", linestyle="--", label="Random (0.50)")
axes[1].axvline(results_df["roc_auc"].median(), color="orange", linestyle="-",
                label=f"Median = {results_df['roc_auc'].median():.3f}")
axes[1].set_xlabel("ROC-AUC")
axes[1].set_ylabel("# Tickers")
axes[1].set_title("Per-Ticker ROC-AUC Distribution")
axes[1].legend()

# --- Scatter: accuracy vs AUC per ticker ---
axes[2].scatter(results_df["roc_auc"], results_df["accuracy"], alpha=0.4, s=15, color="steelblue")
axes[2].axvline(0.5, color="red", linestyle="--", lw=0.8)
axes[2].axhline(0.5, color="red", linestyle="--", lw=0.8)
axes[2].set_xlabel("ROC-AUC")
axes[2].set_ylabel("Accuracy")
axes[2].set_title("Accuracy vs ROC-AUC (per ticker)")

plt.tight_layout()
plt.show()

## Equal-Weighted Portfolio: Signal Applied to Intraday Returns (Test Set)

In [ ]:
# Reconstruct signal and proba wide DataFrames from per-ticker predictions
signal_wide = pd.DataFrame({t: v["predictions"] for t, v in ticker_results.items()})

# Align with intraday returns on the test period
common_dates   = signal_wide.index.intersection(intraday_returns.index)
common_tickers = signal_wide.columns.intersection(intraday_returns.columns)

signal_wide   = signal_wide.loc[common_dates, common_tickers]
intraday_test = intraday_returns.loc[common_dates, common_tickers]

# Equal-weighted long portfolio: stocks where model predicts intraday up
long_mask = signal_wide == 1
portfolio_returns = (
    (intraday_test * long_mask).sum(axis=1)
    / long_mask.sum(axis=1).replace(0, np.nan)
)

benchmark_returns = intraday_test.mean(axis=1)

cum_portfolio = (1 + portfolio_returns).cumprod()
cum_benchmark = (1 + benchmark_returns).cumprod()

def sharpe(r): return r.mean() / r.std() * np.sqrt(252)

print(f"{'':40s}  {'Ann Ret':>9}  {'Ann Vol':>9}  {'Sharpe':>7}")
print(f"{'Signal portfolio (long predicted up)':40s}  {portfolio_returns.mean()*252:>9.2%}  {portfolio_returns.std()*np.sqrt(252):>9.2%}  {sharpe(portfolio_returns):>7.2f}")
print(f"{'Equal-weight benchmark':40s}  {benchmark_returns.mean()*252:>9.2%}  {benchmark_returns.std()*np.sqrt(252):>9.2%}  {sharpe(benchmark_returns):>7.2f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cum_portfolio.index, cum_portfolio.values,
        label="Signal Portfolio (Long Predicted Up)", color="steelblue", lw=1.5)
ax.plot(cum_benchmark.index, cum_benchmark.values,
        label="Equal-Weight Benchmark", color="tomato", linestyle="--", lw=1.5)
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return")
ax.set_title("Per-Ticker XGBoost Signal vs. Benchmark (Test Set)")
ax.legend()
plt.tight_layout()
plt.show()